# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

_Fill in your group number **from Brightspace**, names, and student numbers._
    
|    Group   |           X          |
|------------|----------------------|
| Luca Bixade |        6129080      |
| Student B  |        XXXXXXX       |
| Student C  |        XXXXXXX       |
| Student D  |        XXXXXXX       |

#### Imports

In [766]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""

import numpy as np
import random
import sys
import time

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData

rng = np.random.default_rng(12)
random.seed(6767)

## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

TSP - Traveling Salesman Problem - is a optimisation problem in Computer Science. It's definitions is that, given a list of cities and the distances between all pairs of cities, we have to find the shortest possible route that visits all cities in the given route exactly once and returns to the starting point.

#### Question 2

1. Unlike the basic TSP, our given problem introduces the concept of mazes, walls preventing the travel between specific points.
2. Unlike the basic TSP, our given problem uses different movement, in a grid, instead of geometric distances.
3. Basic TSP starts with the distances already provided, but in this problem, the robot has to use a separate algorithm to find all the paths and distances.

#### Question 3

Since the problem is NP-hard, brute-forcing it is impossible in most cases. Using CI techniques, we can perform heuristic searches, which improve the brute force a lot, since instead of taking all the paths, we only take the most promising ones.

### 1.2 Genetic Algorithm

In [767]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:

    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    """
    def __init__(self, generations, pop_size):
        self.generations = generations
        self.pop_size = pop_size
    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        pass

First, we have to read the data. As the assignment mentioned, we read from optimal_tsp.

In [768]:
from TSPData import TSPData

tsp_data = TSPData.read_from_file("../data/optimal_tsp")

distances = tsp_data.get_distances()       
start_dist = tsp_data.get_start_distances() 
end_dist = tsp_data.get_end_distances() 

print(len(tsp_data.get_distances()))
print(len(tsp_data.get_start_distances()))
print(len(tsp_data.get_end_distances()))

18
18
18


As it can be seen, we receive the correct number of products.

#### Question 4

In my implementation, the genes represent the products that are present in the store, stored using their IDs (the number of the product).
The chromosomes, which are sets of genes, thus, a possible solution or route, are encoded as a list, with each product being saved as its ID-1.

In [769]:
import random


#added pop_size here so it can be tested
def generate_initial_population(num_products, pop_size):
    
    chromosomes = []
    base_route = list(range(num_products))
    
    for _ in range(pop_size):
        random_route = base_route.copy()
        random.shuffle(random_route)
        chromosomes.append(random_route)
        
    return chromosomes

num_products = len(tsp_data.get_start_distances()) 

population = generate_initial_population(num_products, 3)

print(population[0])
print(population[1])
print(population[2])

[12, 2, 5, 17, 14, 11, 6, 13, 3, 15, 16, 0, 8, 1, 7, 10, 4, 9]
[4, 3, 15, 0, 14, 7, 11, 13, 17, 8, 12, 1, 10, 2, 6, 16, 5, 9]
[14, 12, 5, 1, 4, 3, 11, 6, 13, 8, 17, 7, 16, 15, 0, 10, 9, 2]


#### Question 5

For the fitness function, i have chosen to use the inverse distance. 1e-6 was added to it to prevent the stuation of 0 distance, which would imply a division by 0. I have made this decision because, in TSP, we need the smallest distance, and maximize fitness. This means that we flip the way in which distances are perceived, now being direct proportional to fitness, which is more natural.

In [770]:
def calculate_route_distance(chromosome, start_dist, distances, end_dist):
    total_distance = 0.0
    
    first_product = chromosome[0]
    total_distance += start_dist[first_product]
    
    for i in range(len(chromosome) - 1):
        current_product = chromosome[i]
        next_product = chromosome[i+1]
        total_distance += distances[current_product][next_product]
        
    last_product = chromosome[-1]
    total_distance += end_dist[last_product]
    
    return total_distance

def calculate_fitness(distance):
    return 1.0 / (float(distance) + 1e-6)

sample = population[0]
distance = calculate_route_distance(sample, start_dist, distances, end_dist)
fitness = calculate_fitness(distance)

print(f"Route: {sample}")
print(f"Total Distance: {distance}")
print(f"Fitness Score: {fitness}")

Route: [12, 2, 5, 17, 14, 11, 6, 13, 3, 15, 16, 0, 8, 1, 7, 10, 4, 9]
Total Distance: 5299.0
Fitness Score: 0.00018871485182322798


#### Question 6

For the parent selection, i have chosen to use a tournament select, which ensures that the best performer is always selected from a specific number of individuals in the current population, without guaranteeing that it is the absolute best. How it works exactly is that it picks a small subset of individuals, compares their fitness scores, so the inverse of their distances, and takes the highest score(lowest distance).

In [771]:
def tournament_selection(population, fitness_scores, k=3):
    selected_indices = random.sample(range(len(population)), k)
    
    best_index = selected_indices[0]
    best_fitness = fitness_scores[best_index]
    
    for idx in selected_indices:
        if fitness_scores[idx] > best_fitness:
            best_fitness = fitness_scores[idx]
            best_index = idx
            
    return population[best_index]

#### Question 7

For genetic operations, I have chosen to implement a crossover and a mutation swap. For the crossover, we are doing a union of randomly selected subsets of the parents. This is done to, hopefully, get a better offspring. For the mutation swap, we are randomly swapping two genes inside a chromosome, to have diversity. This has also helped against getting stuck in a local minima.

In [772]:
def order_crossover(parent1, parent2):

    size = len(parent1)
    
    start, end = sorted(random.sample(range(size), 2))
    
    child = []

    for i in range(size):
        child.append(-1)

    for i in range(start, end + 1):
        child[i] = parent1[i]
    
    p2_idx= 0
    for i in range(size):
        if child[i] == -1:
            while parent2[p2_idx] in child:
                p2_idx += 1
                
            child[i] = parent2[p2_idx]
            
    return child

In [773]:
def swap(x, y):
    return y, x

In [774]:
def swap_mutation(chromosome, mutation_rate=0.1):
    mutated = chromosome.copy()
    
    if random.random() < mutation_rate:
        idx1, idx2 = random.sample(range(len(mutated)), 2)
        mutated[idx1], mutated[idx2] = swap(mutated[idx1], mutated[idx2])
        
    return mutated

#### Question 8

To avoid getting stuck in a local minima, we have implemented the genetic operation mentioned earlier. Using mutation swap, we ensure that there is still some variance in our chromosomes by slightly altering them (swapping two random genes),  which forces the robot to explore new paths.

#### Question 9

Elitism represents the action of copying the best invidual(s) from the current generation directly into the next one, without performing any changes on them. We have applied it, as we believe this is necessary in our code, because we are already minimising the risk of being stuck in a local minima, and not having it would make our algorithm truly random, which might, in many situations, give us bad final results, by removing the best route.

## Full algorithm:

In [775]:
import random

class GeneticAlgorithm:

    def __init__(self, generations, pop_size, mutation_rate=0.1, use_elitism=True):
        self.generations = generations
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.use_elitism = use_elitism

    def generate_initial_population(self, num_products):
    
        chromosomes = []
        base_route = list(range(num_products))
    
        for _ in range(self.pop_size):
            random_route = base_route.copy()
            random.shuffle(random_route)
            chromosomes.append(random_route)
        
        return chromosomes

    def calculate_route_distance(self, chromosome, start_dist, distances, end_dist):
        total_distance = 0.0
    
        first_product = chromosome[0]
        total_distance += start_dist[first_product]
    
        for i in range(len(chromosome) - 1):
            current_product = chromosome[i]
            next_product = chromosome[i+1]
            total_distance += distances[current_product][next_product]
        
        last_product = chromosome[-1]
        total_distance += end_dist[last_product]
    
        return total_distance

    def calculate_fitness(self, distance):
        return 1.0 / (float(distance) + 1e-6)

    def tournament_selection(self, population, fitness_scores, k=3):
        selected_indices = random.sample(range(len(population)), k)
    
        best_index = selected_indices[0]
        best_fitness = fitness_scores[best_index]
    
        for idx in selected_indices:
            if fitness_scores[idx] > best_fitness:
                best_fitness = fitness_scores[idx]
                best_index = idx
            
        return population[best_index]

    def order_crossover(self, parent1, parent2):

        size = len(parent1)
    
        start, end = sorted(random.sample(range(size), 2))
    
        child = []

        for i in range(size):
            child.append(-1)

        for i in range(start, end + 1):
            child[i] = parent1[i]
    
        p2_idx= 0
        for i in range(size):
            if child[i] == -1:
                while parent2[p2_idx] in child:
                    p2_idx += 1
                
                child[i] = parent2[p2_idx]
            
        return child

    def swap(self, x, y):
        return y, x

    def swap_mutation(self, chromosome, mutation_rate=0.1):
        mutated = chromosome.copy()
    
        if random.random() < mutation_rate:
            idx1, idx2 = random.sample(range(len(mutated)), 2)
            mutated[idx1], mutated[idx2] = self.swap(mutated[idx1], mutated[idx2])
        
        return mutated
    
    def solve_tsp(self, tsp_data):
        
        distances = tsp_data.get_distances()
        start_dist = tsp_data.get_start_distances()
        end_dist = tsp_data.get_end_distances()
        num_products = len(start_dist)
        
        chromosomes = self.generate_initial_population(num_products)
        best_overall_route = None
        best_overall_distance = float('inf')
        
        for gen in range(self.generations):
            fitness_scores = []
            
            for chromoz in chromosomes:
                dist = self.calculate_route_distance(chromoz, start_dist, distances, end_dist)
                fitness = self.calculate_fitness(dist)
                fitness_scores.append(fitness)
                
                if dist < best_overall_distance:
                    best_overall_distance = dist
                    best_overall_route = chromoz
            
            new_population = []
            
            #Elitism
            if self.use_elitism:
                best_idx = fitness_scores.index(max(fitness_scores))
                new_population.append(chromosomes[best_idx].copy())
                
            while len(new_population) < self.pop_size:
                p1 = self.tournament_selection(chromosomes, fitness_scores)
                p2 = self.tournament_selection(chromosomes, fitness_scores)
                
                child = self.order_crossover(p1, p2)
                child = self.swap_mutation(child)
                
                new_population.append(child)
                
            chromosomes = new_population
            
        return best_overall_route

## Test with 1000 runs to ensure that we get the optimal route, and we are not stuck in a local minima (elitism set to true - to get our final value)

In [776]:
pop_size = 5  # very small to have a small time, and also to provide clear information about the effect of elitism.
generations = 200    
read_from_file = "./../data/optimal_tsp"
k = 1000 # number of runs

tsp_data = TSPData.read_from_file(read_from_file)

distances = tsp_data.get_distances()
start_dist = tsp_data.get_start_distances()
end_dist = tsp_data.get_end_distances()

final_route = None
final_distance = float('inf')

for i in range(k):
    ga = GeneticAlgorithm(generations, pop_size, mutation_rate=0.1, use_elitism=True)
    
    current_route = ga.solve_tsp(tsp_data)
    current_distance = ga.calculate_route_distance(current_route, start_dist, distances, end_dist)
    
    if current_distance < final_distance:
        final_distance = current_distance
        final_route = current_route
        print(f"NEW BEST ON RUN {i + 1}| Distance: {final_distance}")
print("")
print(f"Best distance: {final_distance}")
print(f"It's sequence: {final_route}")

NEW BEST ON RUN 1| Distance: 3057.0
NEW BEST ON RUN 2| Distance: 2999.0
NEW BEST ON RUN 5| Distance: 2227.0
NEW BEST ON RUN 13| Distance: 1999.0
NEW BEST ON RUN 77| Distance: 1985.0
NEW BEST ON RUN 183| Distance: 1945.0
NEW BEST ON RUN 540| Distance: 1557.0

Best distance: 1557.0
It's sequence: [0, 1, 4, 6, 13, 15, 3, 8, 5, 12, 14, 7, 17, 9, 11, 10, 2, 16]


## Test with 1000 runs to ensure that we get the optimal route, and we are not stuck in a local minima (elitism set to false, to confirm that we are not stuck in a local minima (by getting a smaller value than with elitism set to true))

In [777]:
import random

class GeneticAlgorithm:

    def __init__(self, generations, pop_size, mutation_rate=0.1, use_elitism=False):
        self.generations = generations
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.use_elitism = use_elitism

    def generate_initial_population(self, num_products):
    
        chromosomes = []
        base_route = list(range(num_products))
    
        for _ in range(self.pop_size):
            random_route = base_route.copy()
            random.shuffle(random_route)
            chromosomes.append(random_route)
        
        return chromosomes

    def calculate_route_distance(self, chromosome, start_dist, distances, end_dist):
        total_distance = 0.0
    
        first_product = chromosome[0]
        total_distance += start_dist[first_product]
    
        for i in range(len(chromosome) - 1):
            current_product = chromosome[i]
            next_product = chromosome[i+1]
            total_distance += distances[current_product][next_product]
        
        last_product = chromosome[-1]
        total_distance += end_dist[last_product]
    
        return total_distance

    def calculate_fitness(self, distance):
        return 1.0 / (float(distance) + 1e-6)

    def tournament_selection(self, population, fitness_scores, k=3):
        selected_indices = random.sample(range(len(population)), k)
    
        best_index = selected_indices[0]
        best_fitness = fitness_scores[best_index]
    
        for idx in selected_indices:
            if fitness_scores[idx] > best_fitness:
                best_fitness = fitness_scores[idx]
                best_index = idx
            
        return population[best_index]

    def order_crossover(self, parent1, parent2):

        size = len(parent1)
    
        start, end = sorted(random.sample(range(size), 2))
    
        child = []

        for i in range(size):
            child.append(-1)

        for i in range(start, end + 1):
            child[i] = parent1[i]
    
        p2_idx= 0
        for i in range(size):
            if child[i] == -1:
                while parent2[p2_idx] in child:
                    p2_idx += 1
                
                child[i] = parent2[p2_idx]
            
        return child

    def swap(self, x, y):
        return y, x

    def swap_mutation(self, chromosome, mutation_rate=0.1):
        mutated = chromosome.copy()
    
        if random.random() < mutation_rate:
            idx1, idx2 = random.sample(range(len(mutated)), 2)
            mutated[idx1], mutated[idx2] = self.swap(mutated[idx1], mutated[idx2])
        
        return mutated
    
    def solve_tsp(self, tsp_data):
        
        distances = tsp_data.get_distances()
        start_dist = tsp_data.get_start_distances()
        end_dist = tsp_data.get_end_distances()
        num_products = len(start_dist)
        
        chromosomes = self.generate_initial_population(num_products)
        best_overall_route = None
        best_overall_distance = float('inf')
        
        for gen in range(self.generations):
            fitness_scores = []
            
            for chromoz in chromosomes:
                dist = self.calculate_route_distance(chromoz, start_dist, distances, end_dist)
                fitness = self.calculate_fitness(dist)
                fitness_scores.append(fitness)
                
                if dist < best_overall_distance:
                    best_overall_distance = dist
                    best_overall_route = chromoz
            
            new_population = []
            
            #Elitism
            if self.use_elitism:
                best_idx = fitness_scores.index(max(fitness_scores))
                new_population.append(chromosomes[best_idx].copy())
                
            while len(new_population) < self.pop_size:
                p1 = self.tournament_selection(chromosomes, fitness_scores)
                p2 = self.tournament_selection(chromosomes, fitness_scores)
                
                child = self.order_crossover(p1, p2)
                child = self.swap_mutation(child)
                
                new_population.append(child)
                
            chromosomes = new_population
            
        return best_overall_route

In [778]:
pop_size = 5  # very small to have a small time, and also to provide clear information about the effect of elitism.
generations = 200    
read_from_file = "./../data/optimal_tsp"
k = 1000 # number of runs

tsp_data = TSPData.read_from_file(read_from_file)

distances = tsp_data.get_distances()
start_dist = tsp_data.get_start_distances()
end_dist = tsp_data.get_end_distances()

final_route = None
final_distance = float('inf')

for i in range(k):
    ga = GeneticAlgorithm(generations, pop_size, mutation_rate=0.1, use_elitism=True)
    
    current_route = ga.solve_tsp(tsp_data)
    current_distance = ga.calculate_route_distance(current_route, start_dist, distances, end_dist)
    
    if current_distance < final_distance:
        final_distance = current_distance
        final_route = current_route
        print(f"NEW BEST ON RUN {i + 1}| Distance: {final_distance}")
print("")
print(f"Best distance: {final_distance}")
print(f"It's sequence: {final_route}")

NEW BEST ON RUN 1| Distance: 2533.0
NEW BEST ON RUN 13| Distance: 1833.0
NEW BEST ON RUN 739| Distance: 1603.0

Best distance: 1603.0
It's sequence: [0, 1, 6, 4, 13, 15, 3, 17, 7, 8, 9, 5, 11, 14, 12, 2, 10, 16]


After testing the algorithm on the same data, both with and without elitism, it is clear that we should use it.

#### Question 10

In [779]:
import random

class GeneticAlgorithm:

    def __init__(self, generations, pop_size, mutation_rate=0.1, use_elitism=False):
        self.generations = generations
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.use_elitism = use_elitism

    def generate_initial_population(self, num_products):
    
        chromosomes = []
        base_route = list(range(num_products))
    
        for _ in range(self.pop_size):
            random_route = base_route.copy()
            random.shuffle(random_route)
            chromosomes.append(random_route)
        
        return chromosomes

    def calculate_route_distance(self, chromosome, start_dist, distances, end_dist):
        total_distance = 0.0
    
        first_product = chromosome[0]
        total_distance += start_dist[first_product]
    
        for i in range(len(chromosome) - 1):
            current_product = chromosome[i]
            next_product = chromosome[i+1]
            total_distance += distances[current_product][next_product]
        
        last_product = chromosome[-1]
        total_distance += end_dist[last_product]
    
        return total_distance

    def calculate_fitness(self, distance):
        return 1.0 / (float(distance) + 1e-6)

    def tournament_selection(self, population, fitness_scores, k=3):
        selected_indices = random.sample(range(len(population)), k)
    
        best_index = selected_indices[0]
        best_fitness = fitness_scores[best_index]
    
        for idx in selected_indices:
            if fitness_scores[idx] > best_fitness:
                best_fitness = fitness_scores[idx]
                best_index = idx
            
        return population[best_index]

    def order_crossover(self, parent1, parent2):

        size = len(parent1)
    
        start, end = sorted(random.sample(range(size), 2))
    
        child = []

        for i in range(size):
            child.append(-1)

        for i in range(start, end + 1):
            child[i] = parent1[i]
    
        p2_idx= 0
        for i in range(size):
            if child[i] == -1:
                while parent2[p2_idx] in child:
                    p2_idx += 1
                
                child[i] = parent2[p2_idx]
            
        return child

    def swap(self, x, y):
        return y, x

    def swap_mutation(self, chromosome, mutation_rate=0.1):
        mutated = chromosome.copy()
    
        if random.random() < mutation_rate:
            idx1, idx2 = random.sample(range(len(mutated)), 2)
            mutated[idx1], mutated[idx2] = self.swap(mutated[idx1], mutated[idx2])
        
        return mutated
    
    def solve_tsp(self, tsp_data):
        
        distances = tsp_data.get_distances()
        start_dist = tsp_data.get_start_distances()
        end_dist = tsp_data.get_end_distances()
        num_products = len(start_dist)
        
        chromosomes = self.generate_initial_population(num_products)
        best_overall_route = None
        best_overall_distance = float('inf')
        
        for gen in range(self.generations):
            fitness_scores = []
            
            for chromoz in chromosomes:
                dist = self.calculate_route_distance(chromoz, start_dist, distances, end_dist)
                fitness = self.calculate_fitness(dist)
                fitness_scores.append(fitness)
                
                if dist < best_overall_distance:
                    best_overall_distance = dist
                    best_overall_route = chromoz
            
            new_population = []
            
            #Elitism
            if self.use_elitism:
                best_idx = fitness_scores.index(max(fitness_scores))
                new_population.append(chromosomes[best_idx].copy())
                
            while len(new_population) < self.pop_size:
                p1 = self.tournament_selection(chromosomes, fitness_scores)
                p2 = self.tournament_selection(chromosomes, fitness_scores)
                
                child = self.order_crossover(p1, p2)
                child = self.swap_mutation(child)
                
                new_population.append(child)
                
            chromosomes = new_population
            
        return best_overall_route

In [780]:
pop_size = 2000
generations = 200   
read_from_file = "./../data/optimal_tsp"
k = 20 # number of runs

tsp_data = TSPData.read_from_file(read_from_file)

distances = tsp_data.get_distances()
start_dist = tsp_data.get_start_distances()
end_dist = tsp_data.get_end_distances()

final_route = None
final_distance = float('inf')

for i in range(k):
    ga = GeneticAlgorithm(generations, pop_size, mutation_rate=0.1, use_elitism=True)
    
    current_route = ga.solve_tsp(tsp_data)
    current_distance = ga.calculate_route_distance(current_route, start_dist, distances, end_dist)
    
    if current_distance < final_distance:
        final_distance = current_distance
        final_route = current_route

    print(f"run {i + 1}: {current_distance}")
print("")
print(f"Best distance: {final_distance}")
print(f"It's sequence: {final_route}")
tsp_data.write_action_file(final_route, "./../data/tsp_solution.txt")

run 1: 1325.0
run 2: 1325.0
run 3: 1325.0
run 4: 1325.0
run 5: 1325.0
run 6: 1325.0
run 7: 1325.0
run 8: 1325.0
run 9: 1325.0
run 10: 1325.0
run 11: 1325.0
run 12: 1325.0
run 13: 1325.0
run 14: 1325.0
run 15: 1325.0
run 16: 1325.0
run 17: 1325.0
run 18: 1325.0
run 19: 1325.0
run 20: 1325.0

Best distance: 1325.0
It's sequence: [0, 1, 6, 4, 13, 15, 3, 8, 7, 17, 9, 14, 11, 12, 5, 10, 2, 16]


As it can be seen in the run above,in 20 runs of 200 generations each, we get the sme result - 1325 (technically 1343, if we add the produts as well), in 100% of the cases. Because of this, i am very inclined to believe that this is the optimal solution, and not just a local minima.

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 12

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 13

<div style="background-color:#f1be3e">

_Write your answer here. You can also use LaTeX notation in a Jupyter Notebook._

#### Question 14

<div style="background-color:#f1be3e">

_Write your answer here. You can also use LaTeX notation in a Jupyter Notebook._

### 2.3 Implementing the Ant Algorithm

In [781]:
# Class that represents the basic Ant functionality
class StandardAnt:

    """
    Constructor of a StandardAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


In [782]:
# Class that holds all of the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ints representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length):
        self.walls = walls
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self):
        pass

    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        pass

    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        pass

    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        pass

    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the poition of interest
    @return the amount of pheromone at the specified poition
    """
    def get_pheromone(self, pos):
        pass

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            return Maze(maze_layout, width, length)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [783]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    """
    Constructs a new optimization object using the ant algorithm
    @param maze: the maze (environment) for ants
    @param ants_per_gen: the number of ants per generation (between update of pheromones)
    @param generations: the total number of generations of ants (pheromone updates)
    @param q: the normalization factor for the amount of dropped pheromone
    @param evaporation: the evaporation factor for the pheromones
    """
    def __init__(self, maze, ants_per_gen, generations, q, evaporation):
        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation

    """
    Loop that starts the shortest path process
    @param path_specification: description of the route we wish to optimize
    @return the optimized route according to the ACO algorithm
    """
    def find_shortest_route(self, path_specification):
        self.maze.reset()
        pass

In [784]:
# Please keep your parameters for the ACO easily changeable here
gen = 1
no_gen = 1
q = 1600
evap = 0.1

# Construct the optimization objects
maze = Maze.create_maze("./../data/hard_maze.txt")
spec = PathSpecification.read_coordinates("./../data/hard_coordinates.txt")
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
print("Route size: " + str(shortest_route.size()))

shortest_route.write_to_file("./../data/hard_solution.txt")

Ready reading maze file ./../data/hard_maze.txt
Time taken: 0.0


AttributeError: 'NoneType' object has no attribute 'size'

### 2.4 Upgrading Your Ants with Intelligence

#### Question 15

In [0]:
# Class that represents the intelligent Ant
class IntelligentAnt:

    """
    Constructor of an IntelligentAnt taking a Maze and PathSpecification
    @param maze: the Maze where the ant will try to find a route
    @param path_specification: the PathSpecification consisting of a start and an end coordinate
    """
    def __init__(self, maze, path_specification):
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position = self.start
        self.rand = random

    """
    Method that performs a single complete run through the maze by the ant
    @return the route found by the ant
    """
    def find_route(self):
        route = Route(self.start)
        pass


<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [0]:
# Please keep your parameters for the synthesis part easily changeable here
gen = 1
no_gen = 1
q = 1000
evap = 0.1

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt")
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#f1be3e">

**If you made use of any non-course resources, cite them below.**